In [1]:
from langgraph.graph import StateGraph
from typing import TypedDict
import os 
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from chat_memory import get_chat_history
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage
from video_processing import analyze_video
from vectorstore_setup import clean_classification_text
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langgraph.graph import StateGraph, START, END
import tempfile

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma(persist_directory="path/to/your/chroma_db", embedding_function=embeddings)

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")


# Whisper speach to text 
from openai import OpenAI
client = OpenAI()




In [2]:
router_prompt = ChatPromptTemplate.from_messages([
    ("system", """Your job is to classify user queries into two buckets: 
     - memory
     - memory and vectorstore
     
     *Guidelines*
     Questions regarding exercises, exercise form, or technique: vectorstore & memory
     General question or statements: memory
   
     *Respond to queries with only the correct class*
     Examples: 
     User: how should I grip the bar for bench press?
     Agent: vectorstore & memory

     User: what muscles should I feel during the Romanian dead lift?
     Agent: vectorstore & memory

     User: How's this? 
     Agent: vectorstore & memory

     User: Is this good squat technique? 
     Agent: vectorstore & memory

     User: Was my bar path better?
     Agent: vectorstore & memory

     User: thakns for your help!
     Agent: memory

"""),

     ("human", "{query}")
     
])

llm = ChatOpenAI(model='gpt-4o')

output_parser = StrOutputParser()

router_chain = router_prompt | llm | output_parser


In [3]:
# Define the state

# the state is a dictionary

class GraphState(TypedDict):

    session_id: int 

    user_query: str

    user_video: str

    classification_image: str

    classified_keywords: str

    top_k_chunks: str

    encoded_images: list[str]

    response: str


workflow = StateGraph(GraphState) 

In [4]:
def video_encoder_node(state: GraphState):
    user_video = state["user_video"]
    user_video_encoded = analyze_video(filepath_in=user_video, frame_count=15, max_seconds=10) # encodes video
    encoded_images = [] # creates a list of encoded images for LLM
    for images in user_video_encoded:
        encoded_images.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{images}"}})
        classification_image = user_video_encoded[0] 
    
    return {"classification_image": classification_image, "encoded_images": encoded_images}


In [5]:
# video classification router NODE

def video_classification_node(state: GraphState):
    classification_image = state["classification_image"]

    router_llm = ChatOpenAI(model='gpt-4o')

    response = router_llm.invoke([

    {"role": "user", "content": 

    [{"type": "text", "text":  """Your job is to analyze images of users working out for proper form, and list the key checkpoints of their to body evaluate. 
    Give me ONLY the bodypart checkpoints. Do NOT include evaluation suggestions. Do NOT include an intro sentence. 
    Output format should be exactly the example below.
    **Example**
    Overhead press

    1. Feet & base
    2. Glutes & legs
    3. Core & Ribcage
    4. Shoulder position
    5. Bar path
    6. Head & Neck
    7. Lockout position
    8. Tempo and control
    """},


        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{classification_image}"}}
        ]}


    ])
    print(response.text)

    classified_keywords = clean_classification_text(response)

    return {"classified_keywords": classified_keywords}


# VectorDB Node

In [ ]:
def vector_db_node(state: GraphState):
    user_query = state["user_query"]
    classified_keywords = state.get("classified_keywords", "")
    retrieval_query = classified_keywords

    if classified_keywords:
        results_user = vectorstore.similarity_search(user_query, k=5)
        results_video = vectorstore.similarity_search(retrieval_query, k=5)
        results = results_user + results_video

    if not classified_keywords: 
        results_user = vectorstore.similarity_search(user_query, k=5)
        results = results_user

    unique_results = []
    seen = set()

    for r in results:
        content = r.page_content
        if content not in seen:
            seen.add(content)
            unique_results.append(r)


    top_k_chunks = "\n".join([r.page_content for r in unique_results])

    print(f"RESULTS COUNT: {len(results)}")
    print(f"UNIQUE RESULTS COUNT: {len(unique_results)}")
    for r in unique_results[:2]:
        print(f"CHUNK PREVIEW: {r.page_content[:100]}")

    return {"top_k_chunks": top_k_chunks}

# Response Generator Node

In [7]:
def response_generator(state: GraphState):
    top_k_chunks = state['top_k_chunks']
    encoded_images = state.get("encoded_images", [])
    session_id = state["session_id"]
    user_query = state["user_query"]

    prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a world-class fitness coach. You have extensive experience in helping weight lifters achieve perfect form and maximum hypertrophy. 
    Your job is to analyze images of users lifting weights, offer them advice from your context, and to answer any questions they might have. 
    Inspect each image CLOSELY and arefuly for problems or issues related to best practices in exercise form. Help the user diagnose their incorrect form. 
    Be specific about what you observe.
             
    # BEHAVIOR INSTRUCTIONS
    1. Tone
    - You're eager and excited to help 
   2. How to analyze
    - Break down your analysis into three sections
        a. What looks good
             - 1 to 2 points
        b. Main fixes
             - Cover all significant issues you observe
        c. Mental cues
             - A brief list of mental cues the lifter can easily remember during their lift


    # ANSWER CONTEXT
    Use ONLY the following context when answering a user: 
        
    ---   
    {top_k_chunks}
    ---
    if the query or image isn't in context, reply, 'I don't have expert coaching content for this exercise yet. 
    Currently I can analyze: bench press, overhead press, incline bench press...'"
    """),
        MessagesPlaceholder(variable_name="history"),
        MessagesPlaceholder(variable_name="input"),
    ])



    user_msg = HumanMessage(content=[
        {"type": "text", "text": user_query},
        *encoded_images,   # <- your list of {"type":"image_url",...}
    ])

    llm = ChatOpenAI(model='gpt-5',
                    temperature=0.5)

    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history=get_chat_history,
        input_messages_key="input",
        history_messages_key="history"

    )

    response = chain_with_history.invoke(
        {"input": [user_msg], "top_k_chunks": top_k_chunks},
        config={"configurable": {"session_id": session_id}}
    )


    return {"response": response}
    

In [8]:
def chat_memory(state: GraphState):
    user_query = state["user_query"]
    session_id = state["session_id"]
    
    prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a world-class fitness coach. You have extensive experience in helping weight lifters achieve perfect form and maximum hypertrophy. 
    Your job is to analyze images of users lifting weights, offer them advice from your context, and to answer any questions they might have. 


    """),
        MessagesPlaceholder(variable_name="history"),
        MessagesPlaceholder(variable_name="input"),
    ])

    user_msg = HumanMessage(content=[
        {"type": "text", "text": user_query}
    ])

    llm = ChatOpenAI(model='gpt-4o',
                    temperature=0.5)

    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history=get_chat_history,
        input_messages_key="input",
        history_messages_key="history"

    )

    response = chain_with_history.invoke(
        {"input": [user_msg]},
        config={"configurable": {"session_id": session_id}}
    )

    return {"response": response}
    


# Router function

In [9]:
def route_query(state: GraphState):
    # first check: is there a video?
    if state["user_video"]:
        return "video_encoder"
    
    # no video — ask the LLM which path
    route = router_chain.invoke({"query": state["user_query"]})
    route = route.lower().strip()
    
    if "vectorstore" in route:
        return "vector_db"
    else:
        return "chat_memory"

# Audio transcription function

In [10]:
# def transcribe_audio(audio_path):
#     audio_file = open(audio_path, "rb") # rb = read binary. audio files are binary data (raw bytes). this tells python gto open as binary instaed of text
#     transcription = client.audio.transcriptions.create(
#         model="whisper-1",
#         file=audio_file
#     )

#     return transcription.text
    

# user_query = transcribe_audio("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/raw_voice_notes/bench_press_voice_note.m4a")
# print(user_query)

In [11]:
# conditional edge
workflow.add_conditional_edges(START, route_query)

# add nodes
workflow.add_node("video_encoder", video_encoder_node)
workflow.add_node("video_classification_router", video_classification_node)
workflow.add_node("vector_db", vector_db_node)
workflow.add_node("response_generator", response_generator)
workflow.add_node("chat_memory", chat_memory)



# connect them
workflow.add_edge("chat_memory", END)
workflow.add_edge("video_encoder", "video_classification_router")
workflow.add_edge("video_classification_router", "vector_db")
workflow.add_edge("vector_db", "response_generator")
workflow.add_edge("response_generator", END)

app = workflow.compile()


In [12]:
result = app.invoke({
    "session_id": 123,
    "user_query": "how's my form?",
    "user_video": "/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/raw_workout_videos/bench_press/bench_02.MP4"
})

print(result["response"])

Frames processed: 15, (9.033333333333333s cap)
Saved to processed-images/bench_02
Video name: bench_02
Bench Press

1. Feet & base
2. Glutes & legs
3. Core & Ribcage
4. Shoulder position
5. Grip & Hand placement
6. Bar path
7. Head & Neck
8. Lockout position
Awesome work getting after it! From these bench press shots, here’s my coach’s eye:

a) What looks good
- You’re taking the bar to your chest with a full range of motion and finishing over your shoulders.
- Tempo looks controlled—no obvious bounce.

b) Main fixes
- Wrists stacked: Your wrists are bent back a lot. Set the bar lower in your palm (over the heel of the hand), squeeze hard, and keep knuckles pointing up so the bar sits directly over your forearm. Wrist wraps can help.
- Elbow angle: Your elbows flare wide near 90°. Tuck them to about 45–60° to your torso to spare shoulders and transfer force better.
- Touch point and forearm verticality: You’re touching a bit high on the chest. Aim 2–3 inches below the nipple line. Adju

In [13]:

# result = app.invoke({
#     "session_id": 123,
#     "user_query": user_query,
#     "user_video": ""
# })

# print(result["response"])

In [14]:
# result = app.invoke({
#     "session_id": 123,
#     "user_query": "thanks for your help today!",
#     "user_video": ""
# })

# print(result["response"])

In [15]:
# result = app.invoke({
#     "session_id": 123,
#     "user_query": "can you tell me again how to make it easier?",
#     "user_video": ""
# })

# print(result["response"])

In [16]:
def correctness_evaluator(run, example):
    generated = run.outputs["answer"]
    reference = example.outputs["answer"]
    question = example.inputs["user_query"]

    prompt = f"""You are evaluating a fitness coaching AI assistant.
Your job is to assess the quality of its response against a reference answer.

Question: {question}
Reference answer: {reference}
Generated answer: {generated}

Scoring criteria:

Score 0 — The generated answer contains incorrect or potentially harmful advice that contradicts the reference answer. This is the worst outcome.
Score 1 — The generated answer is incomplete or omits important details from the reference, but contains nothing incorrect or harmful.
Score 2 — The generated answer is correct and captures the key facts from the reference answer.

Important: Penalize incorrect advice more harshly than omission. A response that says nothing is better than one that says something wrong.

Reply with only the number: 0, 1, or 2."""

    from openai import OpenAI
    openai_client = OpenAI()
    result = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}]
    )
    try: 
        score_raw = result.choices[0].message.content.strip()
        score = int(''.join(filter(str.isdigit, score_raw))[0])
    except:
        score = -1
    return {"key": "correctness", "score": score / 2} # normalize the score so it's <= 1

In [17]:
def hallucination_evaluator(run, example):
    generated = run.outputs["answer"]
    context = run.outputs["context"]
    
    prompt = f"""YYou are evaluating a fitness coaching AI assistant for hallucinations.

This AI analyzes workout VIDEOS and uses retrieved coaching context to provide form advice. 
The AI has two information sources:
1. Retrieved text context (coaching knowledge)
2. Video frames of the user's actual lift (visual observations)

<Instructions>
  - Claims about coaching best practices (e.g. "elbows should be 45-60 degrees") should be verified against the retrieved context
  - Claims about what the user is DOING in their video (e.g. "your elbows are flaring wide") are visual observations and should NOT be penalized for not appearing in the retrieved context
  - Only flag a claim as hallucinated if it contradicts the retrieved context or makes up coaching advice not supported by the context
  - Observations about the user's form that are plausible given a workout video should be treated as valid even if not in the context
</Instructions>

Retrieved context:
{context}

Generated answer:
{generated}

Scoring criteria:

Score 0 — The answer contains coaching advice that directly contradicts the retrieved context, or invents training recommendations with no basis in the context. This is the worst outcome.
Score 1 — The coaching advice is mostly supported by the context but includes minor unsupported recommendations.
Score 2 — All coaching advice and recommendations are directly supported by the retrieved context. Visual observations about the user's form do not count against this score.

Reply with only the number: 0, 1, or 2."""

    from openai import OpenAI
    openai_client = OpenAI()
    result = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}]
    )
    try:
        score_raw = result.choices[0].message.content.strip()
        score = int(''.join(filter(str.isdigit, score_raw))[0])
    except:
        score = None
    return {"key": "hallucination", "score": score}

In [20]:
from uuid import uuid4

def run_rag_pipeline(inputs: dict, attachments: dict) -> dict:
    # grab video from attachments in LangSmith
    video_name = list(attachments.keys())[0] 
    # read the video as raw bytes
    video_bytes = attachments[video_name]["reader"].read()
    
    # save to temp file because analyze_video expects a filepath
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tmp:
        # . delete=False means "don't delete this file when we're done writing" because we still need it for the pipeline.
        tmp.write(video_bytes)
        tmp_path = tmp.name
    
    state = {
        "user_query": inputs["user_query"],
        "session_id": f"eval-{uuid4()}",
        "classified_keywords": "",
        "encoded_images": [],
        "user_video": tmp_path
    }
    
    state = {**state, **video_encoder_node(state)}
    state = {**state, **video_classification_node(state)}
    print(f"BEFORE VECTOR DB - classified_keywords: {state.get('classified_keywords', 'MISSING')}")
    state = {**state, **vector_db_node(state)}
    print(f"AFTER VECTOR DB - top_k_chunks length: {len(state.get('top_k_chunks', ''))}")
    state = {**state, **response_generator(state)}

    print("STATE KEYS:", state.keys())
    print("RESPONSE:", state.get("response", "NOT FOUND"))
    
    return {"answer": state["response"],
            "context": state["top_k_chunks"]}

In [19]:
from langsmith import Client
from langsmith.evaluation import evaluate

langsmith_client = Client()

os.environ["LANGCHAIN_TRACING_V2"] = "false" 
# stops LangSmith from trying to log all the big intermediate data
# evaluate() function will still capture your inputs, outputs, and evaluator scores
# it just won't try to upload the full trace with all those heavy frames.

results = evaluate(
    run_rag_pipeline,
    data="Image assessment response accuracy v1",
    evaluators=[correctness_evaluator, hallucination_evaluator],
    experiment_prefix="vision-faithfulness-rag-v7-structured-output-4sections"
)

View the evaluation results for experiment: 'vision-faithfulness-rag-v7-structured-output-4sections-53c3f656' at:
https://smith.langchain.com/o/5dffb1fc-82e5-4000-a9a5-d9c923e0f73c/datasets/1be4093e-6f7d-4021-a94b-d6cf76c22b6a/compare?selectedSessions=b9161155-4522-4ea7-83d5-31e4620eda94




0it [00:00, ?it/s]

Frames processed: 15, (10s cap)
Saved to processed-images/tmpgn91j2oe
Video name: tmpgn91j2oe
I'm unable to provide feedback on specific individuals' workouts. However, I can certainly share general checkpoints for evaluating form in different exercises. Here's an example for the **Dumbbell Bench Press**:

1. Feet & base
2. Lower back & arch
3. Shoulder blades & back placement
4. Grip & wrist position
5. Elbow path & angle
6. Dumbbell path
7. Head & Neck
8. Breathing and control

BEFORE VECTOR DB - classified_keywords: I m unable to provide feedback on specific individuals  workouts  However  I can certainly share general checkpoints for evaluating form in different exercises  Here s an example for the   Dumbbell Bench Press        Feet   base    Lower back   arch    Shoulder blades   back placement    Grip   wrist position    Elbow path   angle    Dumbbell path    Head   Neck    Breathing and control 
AFTER VECTOR DB - top_k_chunks length: 0
STATE KEYS: dict_keys(['user_query', 'sessi